In [ ]:
# import required classes
import pandas as pd
import glob
import re
from pathlib import Path
import numpy as np        
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from tabulate import tabulate
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import cross_validate, cross_val_score, train_test_split, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, accuracy_score, precision_score, recall_score, make_scorer, f1_score, precision_recall_curve, average_precision_score, roc_auc_score, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer   
from sklearn.pipeline import Pipeline 
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Classifiers
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
#import data files
df_original = pd.read_csv('data/sstaudent_data.csv', sep=',')

# Since data has been previously encoded, load in dictionaries
dictionary = {}
for f in glob.glob("data/dictionaries/*.csv"):
    cat_df = pd.read_csv(f)
    cat_df["combined"] =  cat_df["label"] #cat_df["value"].astype(str) + ": " +
    dictionary[Path(f).stem] = dict(zip(cat_df["value"], cat_df["combined"]))


## 1. EDA
### 1.0 Data prep

In [ ]:
# standardize column names by lowercasing, removing spacial chars, and 
df = df_original.copy()    
new_columns = []

for col in df.columns:
    temp_col = str(col).lower()
    temp_col = temp_col.replace(' ', '_')
    temp_col = re.sub(r'[^a-z0-9_]', '', temp_col)
    new_columns.append(temp_col)
    
df.columns = new_columns

In [ ]:
# create new columns with the translated categorical defintions
decoded_df = df.copy()
decoded_df['marital_status_decoded'] = df['marital_status'].map(dictionary['tA1_marital_status']).astype('category')
decoded_df['application_mode_decoded'] = df['application_mode'].map(dictionary['tA3_application_mode']).astype('category')
decoded_df['course_decoded'] = df['course'].map(dictionary['tA4_course_names']).astype('category')
decoded_df['daytimeevening_attendance_decoded'] = df['daytimeevening_attendance'].map(dictionary['tA9_attendance_regime']).astype('category')
decoded_df['previous_qualification_decoded'] = df['previous_qualification'].map(dictionary['tA5_previous_quals']).astype('category')
decoded_df['nationality_decoded'] = df['application_mode'].map(dictionary['tA2_nationality']).astype('category')
decoded_df['mothers_qualification_decoded'] = df['mothers_qualification'].map(dictionary['tA6_parent_previous_quals']).astype('category')
decoded_df['fathers_qualification_decoded'] = df['fathers_qualification'].map(dictionary['tA6_parent_previous_quals']).astype('category')
decoded_df['mothers_occupation_decoded'] = df['mothers_occupation'].map(dictionary['tA7_parent_occupation']).astype('category')
decoded_df['fathers_occupation_decoded'] = df['fathers_occupation'].map(dictionary['tA7_parent_occupation']).astype('category')
decoded_df['gender_decoded'] = df['gender'].map(dictionary['tA8_gender']).astype('category')
decoded_df['displaced_decoded'] = df['displaced'].map(dictionary['tA10_yes_no']).astype('category')
decoded_df['educational_special_needs_decoded'] = df['educational_special_needs'].map(dictionary['tA10_yes_no']).astype('category')
decoded_df['tuition_fees_up_to_date_decoded'] = df['tuition_fees_up_to_date'].map(dictionary['tA10_yes_no']).astype('category')
decoded_df['debtor_decoded'] = df['debtor'].map(dictionary['tA10_yes_no']).astype('category')
decoded_df['scholarship_holder_decoded'] = df['scholarship_holder'].map(dictionary['tA10_yes_no']).astype('category')
decoded_df['international_decoded'] = df['international'].map(dictionary['tA10_yes_no']).astype('category')



In [ ]:
# Define numerical variables
noncategorical_cols = ['age_at_enrollment', 'application_order',
'curricular_units_1st_sem_credited', 'curricular_units_1st_sem_enrolled',
'curricular_units_1st_sem_evaluations', 'curricular_units_1st_sem_approved',
'curricular_units_1st_sem_grade', 'curricular_units_1st_sem_without_evaluations',
'curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled',
'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved',
'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations',
'unemployment_rate', 'inflation_rate', 'gdp']

# Make lists for  categorical columns
categorical_cols_orginal = ['marital_status', 'application_mode', 'course',
       'daytimeevening_attendance', 'previous_qualification', 'nationality',
       'mothers_qualification', 'fathers_qualification', 'mothers_occupation',
       'fathers_occupation', 'displaced', 'educational_special_needs',
       'debtor', 'tuition_fees_up_to_date', 'gender', 'scholarship_holder',
       'international']

# Make lists for  categorical columns with decoded values
categorical_cols_decoded = ['marital_status_decoded', 'application_mode_decoded', 'course_decoded',
       'daytimeevening_attendance_decoded', 'previous_qualification_decoded', 'nationality_decoded',
       'mothers_qualification_decoded', 'fathers_qualification_decoded', 'mothers_occupation_decoded',
       'fathers_occupation_decoded', 'displaced_decoded', 'educational_special_needs_decoded',
       'debtor_decoded', 'tuition_fees_up_to_date_decoded', 'gender_decoded', 
       'scholarship_holder_decoded', 'international_decoded']

# convert col type to categorical
for i, col in enumerate(categorical_cols_orginal):
   df[col] = df[col].astype('category')
   decoded_df[col] = decoded_df[col].astype('category')


### 1.1 Shape of DataFrame

In [ ]:

print(decoded_df.info())

### 1.2 Missing Values

In [ ]:
#check null rows - inverse of non-null values
null_counts = decoded_df.isnull().sum()
null_counts = null_counts[null_counts > 0]
print(null_counts)

### 1.3 Non-Categorical Variables

In [ ]:
# Create Boxplots for Non-Categorical Values
fig, axes = plt.subplots(ncols=3, nrows=6, figsize=(10, 15))
x_counter = 0
y_counter = 0
for i, col in enumerate(noncategorical_cols):
    sns.boxplot(data=decoded_df, x=col, y='target', ax=axes[y_counter, x_counter])
    x_counter += 1
    if x_counter == 3:
        x_counter = 0
        y_counter += 1
    if i == len(noncategorical_cols) :
        break

fig.suptitle("Figure 1.1: Boxplots Original Df Continuous Vars", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# summary of non-categorical variables
print('\nSummary metrics for numerical fields | Original Df')
print(decoded_df.describe())

for i, col in enumerate(noncategorical_cols):
        
    top_values = decoded_df.nlargest(5, col)
    
    print(f'\nTop 5 `{col}`', top_values[col])
    Q1 = decoded_df[col].quantile(0.25)
    Q3 = decoded_df[col].quantile(0.75)
    upper_whisker =  Q3 + 1.5 * (Q3 - Q1)
    
    print(f'\nUpper Whisker: `{col}`', upper_whisker)
    count_above_whisker = (decoded_df[col] > upper_whisker).sum()
    print("Count above upper whisker:", count_above_whisker)

### 1.4 - Categorical Variables
Since data has been previously encoded, we build a dictionary to translate the encoding to the categorical definitions

In [ ]:
#  print value counts for each  column for our decoded categorical variables
       
print('\nCategorical Values by Column:')
for i, col in enumerate(categorical_cols_decoded):

    #decoded_df[col] = decoded_df[col].astype('category')
    order = decoded_df[col].value_counts().index
    decoded_df[col] = pd.Categorical(decoded_df[col], categories=order, ordered=True)


    sns.histplot(data=decoded_df, x=col, hue='target',  multiple="stack", shrink=0.8)
    plt.title(f'Figure 1.{i + 2}: Bar chart for {col}')
    plt.xticks(rotation=90)
    plt.show()

    decoded_df[col] = decoded_df[col].astype('category')
    val_counts = decoded_df[col].value_counts()
    val_percent = decoded_df[col].value_counts(normalize=True) * 100

    summary = pd.DataFrame({
        'Count': val_counts,
        'Percentage': val_percent.round(2) 
    })
    
    print(f"\nValues for DF column : {col}")
    print(summary)

#### 1.4.1 Possible Outliers
* One person age 70, where rest between 17-62. We can probably ignore or bin by age
* Possible data entry error in application_order. Only one "9", rest range from 1-6, with IQR between 1-2.


### 1.5 Check for Correlation

In [ ]:
# Pearson Correlation Coefficitent Test
X = decoded_df[noncategorical_cols]

plt.figure(figsize=(14,10))
corr = X.corr()
sns.heatmap(corr, cmap="BrBG", annot=False)
plt.title('\n Figure 1.19: Correlation Heatmap')

print('\n Table 1: Pearson Correlation Coefficients')
# print values
corr_abs_sorted = corr.abs().unstack().sort_values(ascending=False)
corr_abs_sorted = corr_abs_sorted[corr_abs_sorted < 1]
print(corr_abs_sorted.head(20).to_markdown())

In [ ]:
# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)

print('\n Table 2: VIF')
print(vif_data.to_markdown())

### 1.6 Check for Imbalance 

Our data shows some imbalance. Ratio is closer if we do not count enrolled.

In [ ]:
val_counts = decoded_df['target'].value_counts()
val_percent = decoded_df['target'].value_counts(normalize=True) * 100

print(pd.DataFrame({
    'Count': val_counts,
    'Percentage': val_percent.round(2) 
}))


print('Percent dropped out (if enrolled not counted):')
print(1338 / (1338 + 2129) * 100 )


## 2. Feature Engineering

### 2.1.1 Impute missing values

The dataset had one missing value for `fathers_occupation_decoded`. A value was provided for `fathers_occupation` but doesn't match any in our dictionary so will convert to "Other". 

In [ ]:
# Replace missing values with "Other"
decoded_df['fathers_occupation_decoded'] = decoded_df['fathers_occupation_decoded'].cat.add_categories(['Other'])
decoded_df['fathers_occupation_decoded'] = decoded_df['fathers_occupation_decoded'].fillna('Other')

#### 2.1.2 Drop rows for students without any academic information

180 of 4424 (~4%) observations have not values entered into any of the 12 1st and 2nd semester fields fields, but had different outcomes/targets. I will drop since this indicates that the student was not in school during the period of interest and records could just add noise to our models.

In [ ]:
# drop rows that don't include any values in all of the following fields
curricular_unit_cols = ['curricular_units_1st_sem_credited', 'curricular_units_1st_sem_enrolled',
'curricular_units_1st_sem_evaluations', 'curricular_units_1st_sem_approved',
'curricular_units_1st_sem_grade', 'curricular_units_1st_sem_without_evaluations',
'curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled',
'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved',
'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations']

missing_vals_df = (decoded_df[curricular_unit_cols].eq(0).all(axis=1))
print(missing_vals_df.sum())

decoded_engineered_df =  decoded_df[~missing_vals_df].copy()

2.1.3 Combining fields

Create new engineered fields for total_credited, total_enrolled, total_evaluations, and total_approved with the sums of the respective values for te 1st and 2nd semester. Create weighted avg grade of both semesters' grades.

In [ ]:
# combine 1st and 2nd semester variables
decoded_engineered_df["total_credited"] = decoded_engineered_df["curricular_units_1st_sem_credited"] + decoded_engineered_df["curricular_units_2nd_sem_credited"]
decoded_engineered_df["total_enrolled"] = decoded_engineered_df["curricular_units_1st_sem_enrolled"] + decoded_engineered_df["curricular_units_2nd_sem_enrolled"]
decoded_engineered_df["total_evaluations"] = decoded_engineered_df["curricular_units_1st_sem_evaluations"] + decoded_engineered_df["curricular_units_2nd_sem_evaluations"]
decoded_engineered_df["total_approved"] = decoded_engineered_df["curricular_units_1st_sem_approved"] + decoded_engineered_df["curricular_units_2nd_sem_approved"]

# omit total_wout_evaluations, since this information is captured in _enrolled and _evaluations
#decoded_engineered_df["total_wout_evaluations"] = decoded_engineered_df["curricular_units_1st_sem_without_evaluations"] + decoded_engineered_df["curricular_units_2nd_sem_without_evaluations"]

# create weighted grad average
g1 = decoded_engineered_df["curricular_units_1st_sem_grade"]
g2 = decoded_engineered_df["curricular_units_2nd_sem_grade"]
n1 = decoded_engineered_df["curricular_units_1st_sem_evaluations"]  
n2 = decoded_engineered_df["curricular_units_2nd_sem_evaluations"]
decoded_engineered_df["weighted_avg_grade"] = (
    (g1 * n1 + g2 * n2) / (n1 + n2).replace(0, np.nan)
).fillna(0)



### 2.2 Dropping Features

We will drop columns that are highly correlated. For example 1st and 2nd enrolled

In [ ]:
# drop original curricular fields 
decoded_engineered_df.drop(columns=curricular_unit_cols, inplace=True)
decoded_engineered_df.drop(columns=['unemployment_rate','application_order'], inplace=True)


# drop original categorical encoded fields, as we will be re-coding them using one hot encoding
decoded_engineered_df.drop(columns=categorical_cols_orginal, inplace=True)

In [ ]:
# Kernel Density for new set of numerical fields
numeric_cols = decoded_engineered_df.select_dtypes(include=['int64', 'float64']).columns

fig, axes = plt.subplots(ncols=3, nrows=3, figsize=(10, 8))
x_counter = 0
y_counter = 0
for i, col in enumerate(numeric_cols):
    sns.histplot(data=decoded_engineered_df, x=col, hue='target', kde=True, stat="density", ax=axes[y_counter, x_counter])
    x_counter += 1
    if x_counter == 3:
        x_counter = 0
        y_counter += 1
    if i == len(numeric_cols) :
        break

fig.suptitle("Figure 2.1: Kernel Density Plots of Continuous Vars", fontsize=14)
plt.tight_layout()
plt.show()


print('\nSummary metrics for numerical fields | Original Df')
print(decoded_engineered_df.describe())

In [ ]:
# calculate correlation on new params
X = decoded_engineered_df[numeric_cols]
corr = X.corr()
print('\n Correlation Heatmap | Extended Df')

# print values
corr_abs_sorted = corr.abs().unstack().sort_values(ascending=False)
corr_abs_sorted = corr_abs_sorted[corr_abs_sorted < 1]
print(corr_abs_sorted.head(20))


In [ ]:

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
print(vif_data)



### 2.3 Grouping categories

The following section groups similar categories together to assist with dimensionality reduction.

In [ ]:
# Group categories for marital status
combine_map = {
    "Married": "Married",
    "Widower": "Other",
    "Divorced": "Other",
    "Common-law marriage": "Married",
    "Legally separated": "Other",
}

# Apply the mapping to create a new column
decoded_engineered_df["marital_status_decoded"] = decoded_engineered_df["marital_status_decoded"].replace(combine_map)

print(decoded_engineered_df["marital_status_decoded"].unique())


In [ ]:
# Group categories for application mode
col = "application_mode_decoded"

combine_map = {
    "Change in course": "Change in institution/course",
    "Transfer": "Change in institution/course",
    "Change in institution/course": "Change in institution/course",
    "International student (bachelor)": "International student",
    "Change in institution/course (International)": "International student",
    "1st phase—general contingent": "1st phase",
    "1st phase—special contingent (Madeira Island)": "1st phase",
    "1st phase—special contingent (Azores Island)": "1st phase",
    "2nd phase—general contingent": "2nd phase",
    "3rd phase—general contingent": "3rd phase",
    "Technological specialization diploma holders": "Specialized diploma",
    "Short cycle diploma holders":  "Specialized diploma"
}
decoded_engineered_df[col] = decoded_engineered_df[col].replace(combine_map)

# group all others with less than 5% of samples
val_counts = decoded_engineered_df[col].value_counts()
val_percent = decoded_engineered_df[col].value_counts(normalize=True) * 100
to_group = val_percent[val_percent < 2].index
decoded_engineered_df[col] = decoded_engineered_df[col].replace(to_group, "Other")

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))



In [ ]:
# Group categories for major
col = "course_decoded"

combine_map = {
    "Management (evening attendance)": "Management",
    "Social Service (evening attendance)": "Social Service",
    "Basic Education": "Other",
    "Biofuel Production Technologies": "Other",
    "Communication Design": "Design and Multimedia",
    "Animation and Multimedia Design": "Design and Multimedia"
}

decoded_engineered_df[col] = decoded_engineered_df[col].replace(combine_map)

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))

In [ ]:
# Group categories for prev qualification
col = "previous_qualification_decoded"

combine_map = {
    "Higher education—degree": "Higher education" ,
    "Higher education—bachelor’s degree": "Higher education",
    "Higher education—degree (1st cycle)": "Higher education",
    "Higher education—master’s degree": "Higher education",
    "Higher education—master’s degree (1st cycle)": "Higher education",
    "Higher education—master’s degree (2nd cycle)": "Higher education" ,
    "Higher education—doctorate": "Higher education" ,
    "Frequency of Higher Education": "Higher education",
    "Technological specialization course": "Specialized course",
    "Professional higher technical course": "Specialized course",
}

decoded_engineered_df[col] = decoded_engineered_df[col].replace(combine_map)
    
val_counts = decoded_engineered_df[col].value_counts()
val_percent = decoded_engineered_df[col].value_counts(normalize=True) * 100
to_group = val_counts[val_counts <= 100].index
decoded_engineered_df[col] = decoded_engineered_df[col].replace(to_group, "Other")

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))



In [ ]:
# Group categories for nationality
col = "nationality_decoded"
 
val_counts = decoded_engineered_df[col].value_counts()
val_percent = decoded_engineered_df[col].value_counts(normalize=True) * 100
to_group = val_counts[val_percent < 2].index
decoded_engineered_df[col] = decoded_engineered_df[col].replace(to_group, "Other")

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))

In [ ]:
# Group categories for parent's qualifications
col = "mothers_qualification_decoded"

combine_map = {
    "Higher Education—degree": "Higher education" ,
    "Higher Education—bachelor’s degree": "Higher education",
    "Higher Education—degree (1st cycle)": "Higher education",
    "Higher education—degree (1st cycle)": "Higher education",
    "Higher Education—master’s degree": "Higher education",
    "Higher Education—master’s degree (1st cycle)": "Higher education",
    "Higher Education—master’s degree (2nd cycle)": "Higher education" ,
    "Higher Education—doctorate": "Higher education",
    "Higher Education—doctorate (3rd cycle)": "Higher education",
    "Frequency of Higher Education": "Higher education",
    "2nd cycle of the general high school course": "Some High School",
    "Other—11th Year of Schooling": "Some High School",
    "12th Year of Schooling—not completed": "Some High School",
    "11th Year of Schooling—not completed": "Some High School",
    "10th Year of Schooling": "Some High School",
    "Complementary High School Course—not concluded": "Some High School",
    "2nd year complementary high school course": "Some High School",
    "Complementary High School Course": "Some High School",
    "9th Year of Schooling—not completed": "Some High School",
    "7th Year (Old)": "Basic education",
    "7th year of schooling": "Basic education",
    "8th year of schooling": "Basic education",
    "General Course of Administration and Commerce": "Specialized course",
    "General commerce course": "Specialized course",
    "Technological specialization course": "Specialized course",
    "Technical-professional course": "Specialized course",
    "Professional higher technical course": "Specialized course",
    "Specialized higher studies course": "Specialized course",
    "Supplementary Accounting and Administration": "Specialized course",
    "Can read without having a 4th year of schooling": "Basic education or less",
    "Cannot read or write": "Basic education or less",
    "Secondary Education—12th Year of Schooling or Equivalent": "Secondary Education"
}

decoded_engineered_df[col] = decoded_engineered_df[col].replace(combine_map)
decoded_engineered_df[col] = decoded_engineered_df[col].apply(
    lambda x: "Basic education or less" if "basic education" in str(x).lower() else x
)

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))

col = "fathers_qualification_decoded"
decoded_engineered_df[col] = decoded_engineered_df[col].replace(combine_map)
decoded_engineered_df[col] = decoded_engineered_df[col].apply(
    lambda x: "Basic education or less" if "basic education" in str(x).lower() else x
)

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))


In [ ]:
# Group categories for parent's occupation
col = "mothers_occupation_decoded"

val_percent = decoded_engineered_df[col].value_counts(normalize=True) * 100
to_group = val_percent[val_percent < 5].index
decoded_engineered_df[col] = decoded_engineered_df[col].replace(to_group, "Other")

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))

col = "fathers_occupation_decoded"

val_percent = decoded_engineered_df[col].value_counts(normalize=True) * 100
to_group = val_percent[val_percent < 5].index
decoded_engineered_df[col] = decoded_engineered_df[col].replace(to_group, "Other")

print(pd.DataFrame({'Count': decoded_engineered_df[col].value_counts(normalize=True) }))

In [ ]:
# print facetted barplots for new categorical values

max_cols = 3
max_rows = 6
fig_ori_dist, axes = plt.subplots(ncols=max_cols, nrows=max_rows, figsize=(15, 20))
categorical_cols_orginal.append('target')

x_counter = 0
y_counter = 0
for i, col in enumerate(categorical_cols_decoded):

    decoded_engineered_df[col] = decoded_engineered_df[col].astype('category')

    sns.histplot(data=decoded_engineered_df, x=decoded_engineered_df[col].astype(str), hue='target', ax=axes[y_counter, x_counter], multiple="stack")
    axes[y_counter, x_counter].tick_params(axis='x', labelsize=8, labelrotation=90)
    axes[y_counter, x_counter].get_legend().remove()
    x_counter += 1
    if x_counter == max_cols:
        x_counter = 0
        y_counter += 1
    if i == len(categorical_cols_decoded) :
        break

fig_ori_dist.suptitle("Figure 2.2: Counts for Df Categorical Vars", fontsize=14)
plt.tight_layout(pad=1)
plt.show()



### 2.4 What about Enrolled

18% of our observations are marked as in "enrolled", which is the student's current state. Keeping this class might make it harder to find the signals for dropout as students my drop out later. Therefore, we will reserve these observations as our inference dataset

In [ ]:
# hold back enrolled students to reduce possible noise and use as inference df later
inference_df = decoded_engineered_df[decoded_engineered_df["target"] == 'Enrolled'].copy()
inference_df.drop(columns=['target'], inplace=True)


simplified_df = decoded_engineered_df[decoded_engineered_df["target"] != 'Enrolled'].copy()
# copy
simplified_df['y'] = simplified_df['target'].replace({'Graduate': 0, 'Dropout': 1}).astype('category')
simplified_df.drop(columns=['target'], inplace=True)

val_counts = simplified_df['y'].value_counts()
val_percent = simplified_df['y'].value_counts(normalize=True) * 100

print(pd.DataFrame({
    'Count': val_counts,
    'Percentage': val_percent.round(2) 
}))

In [ ]:
print(simplified_df.columns)
simplified_df.to_csv("proccessed_dataset.csv", index=False)

## 3. Model Training

### 3.0 - Base Model (Decision Tree)

In [ ]:
# split our data
X = simplified_df.drop(['y'], axis=1)
y = simplified_df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13, stratify=y)

# reset categorical_cols
categorical_cols = categorical_cols_decoded

# define our preprocessor
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
], remainder='drop')

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=13))
])

# Fit
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)


In [ ]:
# plot unpruned decision tree
categorical_features = pipeline.named_steps['preprocessor']\
                .named_transformers_['cat']\
                .get_feature_names_out(categorical_cols)

plt.figure(figsize=(20,20))
plot_tree(
    pipeline.named_steps['classifier'],
    feature_names=list(numeric_cols) + list(categorical_features),
    class_names=pipeline.named_steps['classifier'].classes_.astype(str),
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title('Figure 3.1: Unpruned Decision Tree')
plt.show()

In [ ]:
max_depths = range(1, 21)
cv_scores = []

for depth in max_depths:
   
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=depth, random_state=13))
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1_macro') 
    cv_scores.append(np.mean(scores))

# Find the best depth
best_depth = max_depths[np.argmax(cv_scores)]
print("Best max_depth:", best_depth)

# Optional: plot performance vs depth
import matplotlib.pyplot as plt

plt.plot(max_depths, cv_scores, marker='o')
plt.title('Figure 3.2: Decision Tree CV Max Depths')
plt.xlabel('max_depth')
plt.ylabel('Cross-validated F1 score')
plt.show()

In [ ]:
dt_model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=7, random_state=13))
    ])
dt_model.fit(X_train, y_train)
print("Train score:", dt_model.score(X_train, y_train))

plt.figure(figsize=(20,20))
plot_tree(
    dt_model.named_steps['classifier'],
    feature_names=list(numeric_cols) + list(categorical_features),
    class_names=dt_model.named_steps['classifier'].classes_.astype(str),
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Figure 3.3: Decision Tree with Max Depth')
plt.show()


In [ ]:
# Evaluation Metrics
eval_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Prec', 'Recall', 'F1', 'AUC', 'AUPRC'])

# helper to create evaluation table
def append_eval(model_name, y_true, preds, probs):

    return  pd.concat([eval_df, pd.DataFrame([{
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, preds),
        'Prec': precision_score(y_true, preds, zero_division=0),
        'Recall': recall_score(y_true, preds, zero_division=0),
        'F1': f1_score(y_true, preds, zero_division=0),
        'AUC': roc_auc_score(y_true, probs),
        'AUPRC': average_precision_score(y_true, probs)
    }])], ignore_index=True)


def evaluate_model(model, model_name):
    global X_test, y_test, eval_df

    print(f'\n{model_name}\n'),

    # print matrix
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f'Confusion Matrix for {model_name}\n', confusion_matrix(y_test, y_pred))
    print(f"\nClassification Report for {model_name}:\n", classification_report(y_test, y_pred))

    # print top ten
    clf = model.named_steps['classifier']
    # and callable(getattr(clf, 'feature_importances_'))
    if (hasattr(clf, 'feature_importances_')):
        preprocessor = model.named_steps['preprocessor']
        features = pd.DataFrame(
            clf.feature_importances_,
            index=preprocessor.get_feature_names_out(),
            columns=['importance']
        )

        print(f'\nTop Features \n', features.sort_values(by='importance', ascending=False).head(20))


    return append_eval(model_name, y_test, y_pred, y_proba)



In [ ]:
eval_df = evaluate_model(dt_model, model_name="Decision Tree")

### 3.1 Random Forest

Chose Random Forest on account of its ability to 

In [ ]:
# degine preprocessor for rf
preprocessor2 = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('standardizer', StandardScaler())
        ]), numeric_cols)
    ], remainder='drop'
)



# Build base model
rf0_model = ImbPipeline([
    ('preprocessor', preprocessor2),
    ('smote', SMOTE(random_state=13)),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=13,
        oob_score=True,
        n_jobs=-1
    ))
])

rf0_model.fit(X_train, y_train)

scores = cross_val_score(rf0_model, X_train, y_train, cv=10, scoring='f1_macro')
print("Mean F1-macro CV score:", scores.mean())

eval_df = evaluate_model(rf0_model, model_name="RF0")


categorical_features = rf0_model.named_steps['preprocessor']\
                .named_transformers_['cat']\
                .get_feature_names_out(categorical_cols)

In [ ]:
# loop through list of estimators and perform 5 k-Fold cross validation

oobScores_gini = []
oobScores_ent = []
oobScores_cv = []
oobScores_cv_ent = []

n_estimator = list(range(50, 501, 50))

for n in n_estimator:
    
    rf_pipe = ImbPipeline([
         ('preprocessor', preprocessor2),
        ('smote', SMOTE(random_state=13)),
        ('classifier', RandomForestClassifier(
            n_estimators=n,
            random_state=13,
            oob_score=True,
            max_depth=23, 
            min_samples_leaf=2, 
            min_samples_split=7,
            n_jobs=-1
        ))
    ])

    rf_pipe_ent = ImbPipeline([
         ('preprocessor', preprocessor2),
        ('smote', SMOTE(random_state=13)),
        ('classifier', RandomForestClassifier(
            n_estimators=n,
            criterion='entropy', 
            random_state=13,
            oob_score=True,
            max_depth=23, 
            min_samples_leaf=2, 
            min_samples_split=7,
            n_jobs=-1
        ))
    ])

    # Train our model Using 'gini'
    rf_pipe.fit(X_train, y_train)
    oobScores_gini.append(rf_pipe.named_steps['classifier'].oob_score_)

    # Cross Validate
    scores = cross_val_score(rf_pipe, X_train, y_train, cv=10)
    oobScores_cv.append(scores.mean())

    # Train our model Using 'entropy'
    rf_pipe_ent.fit(X_train, y_train)
    oobScores_ent.append(rf_pipe_ent.named_steps['classifier'].oob_score_)

    # Cross Validate
    scores = cross_val_score(rf_pipe_ent, X_train, y_train, cv=10)
    oobScores_cv_ent.append(scores.mean())




In [ ]:
# plot results from previous step
plt.figure(figsize=(8, 5))
plt.plot(n_estimator, oobScores_gini, color='red', marker='o',  label='Gini OOB Score', linewidth=2)
plt.plot(n_estimator, oobScores_cv,  color='green', marker='v', label='Gini 5-Fold CV Score', linewidth=2)
plt.plot(n_estimator, oobScores_ent, color='blue', marker='o',  label='Entropy OOB Score', linewidth=2)
plt.plot(n_estimator, oobScores_cv_ent,  color='black', marker='v', label='Entropy 5-Fold CV Score', linewidth=2)
plt.title('Figure 3.4 - Random Forest: OOB vs Cross-Validation Performance')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:

# print comparison table
cv_df = pd.DataFrame({ 'n_estimator' : n_estimator, 'Gini 5-Fold CV Score' : oobScores_cv })
gini_df = pd.DataFrame({ 'n_estimator' : n_estimator, 'Gini' : oobScores_gini })
ent_df = pd.DataFrame({ 'n_estimator' : n_estimator, 'Ent' : oobScores_ent })
ent_cv_df = pd.DataFrame({ 'n_estimator' : n_estimator, 'Ent 5-Fold CV Score' : oobScores_cv_ent })
oob_scores = pd.merge(cv_df, ent_cv_df, left_on='n_estimator', right_on='n_estimator', how='left')
oob_scores = pd.merge(ent_df, oob_scores, left_on='n_estimator', right_on='n_estimator', how='left')
oob_scores = pd.merge(gini_df, oob_scores, left_on='n_estimator', right_on='n_estimator', how='left')
print('\nTable 3.1: OBB Score Differences by criterion \n', oob_scores)


In [ ]:
# retrain model using max depth from above

rf0_pipe = ImbPipeline([
    ('preprocessor', preprocessor2),
    ('smote', SMOTE(random_state=13)),
    ('rf', RandomForestClassifier(
        n_estimators=400,
        criterion='entropy', 
        random_state=13,
        oob_score=True,
        n_jobs=-1
    ))
])


# lets get the best depth
param_dist = {
    'rf__max_depth': randint(2, 40),           
    'rf__min_samples_split': randint(2, 20),
    'rf__min_samples_leaf': randint(1, 10),
    'rf__max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=rf0_pipe,
    param_distributions=param_dist,
    n_iter=40,                     
    scoring='f1_macro',          
    cv=5,
    random_state=13,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best score:", random_search.best_score_)
best_depth = random_search.best_params_['rf__max_depth']

In [ ]:
# train using the above
rf_model = ImbPipeline([
    ('preprocessor', preprocessor2),
    ('smote', SMOTE(random_state=13)),
    ('classifier', RandomForestClassifier(
        n_estimators=400,
        criterion='entropy', 
        random_state=13,
        oob_score=True,
        max_depth=38, 
        min_samples_leaf=2, 
        min_samples_split=5,
        n_jobs=-1
    ))
])


rf_model.fit(X_train, y_train)

In [ ]:
# Evaluate Model
eval_df = evaluate_model(rf_model, model_name="RF1")

### 3.2 XGBoost

In [ ]:
# XGBoost

xgb0_pipe = ImbPipeline([
    ('preprocessor', preprocessor2),
    ('smote', SMOTE(random_state=13)),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=13
    ))
])

xgb0_pipe.fit(X_train, y_train)

scores = cross_val_score(xgb0_pipe, X_train, y_train, cv=10, scoring='f1_macro')
print("Mean F1-macro CV score:", scores.mean())

# Evaluate Model
eval_df = evaluate_model(xgb0_pipe, model_name="XG0")

In [ ]:
# Use RandomSearch to Tune 
param_dist = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [3, 4, 5, 6, 8],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__subsample': [0.6, 0.7, 0.8, 1.0],
    'classifier__colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'classifier__min_child_weight': [1, 3, 5, 7],
    'classifier__gamma': [0, 0.1, 0.3, 0.5],
    'classifier__reg_lambda': [1, 1.5, 2, 5, 10],
    'classifier__reg_alpha': [0, 0.1, 0.5, 1],
}


random_search = RandomizedSearchCV(
    estimator=xgb0_pipe,
    param_distributions=param_dist,
    n_iter=20,                
    scoring='f1_macro',      
    cv=10,
    verbose=2,
    n_jobs=-1,
    random_state=13
)

# === Fit Search ===
random_search.fit(X_train, y_train)

print("Best score:", random_search.best_score_)
print("Best params:", random_search.best_params_)



In [ ]:
# Evaluate Model
xgb1_model = random_search.best_estimator_
eval_df = evaluate_model(xgb1_model, model_name="XG1")

In [ ]:
# Test different learning rates
from sklearn.base import clone

results = []
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]

for lr in learning_rates:

    model = clone(xgb1_model)
    model.set_params(classifier__learning_rate=lr)

    scores = cross_validate(
        model,
        X_train, y_train,
        cv=5,
        scoring={
            "accuracy": "accuracy",
            "recall_yes": make_scorer(recall_score, pos_label=1),
            "f1_yes": make_scorer(f1_score, pos_label=1)
        },
        n_jobs=-1
    )
    
    results.append({
        "learning_rate": lr,
        "accuracy": scores["test_accuracy"].mean(),
        "recall_yes": scores["test_recall_yes"].mean(),
        "f1_yes": scores["test_f1_yes"].mean()
    })

df_lr = pd.DataFrame(results).sort_values("learning_rate", ascending=True)
print(df_lr)

In [ ]:
# Plot learning rates
plt.figure(figsize=(10,6))
plt.plot(df_lr["learning_rate"], df_lr["f1_yes"], marker="o")
plt.xscale("log")
plt.xlabel("Learning Rate (log scale)")
plt.ylabel("F1 (Yes)")
plt.title("Figure 3.3: F1 Score vs Learning Rate")
plt.grid(True)
plt.show()


In [ ]:
xgb2_model = clone(xgb1_model)
xgb2_model.set_params(classifier__learning_rate=0.050)
xgb2_model.fit(X_train, y_train)
eval_df = evaluate_model(xgb2_model, model_name="XG2")


### 3.3 SVM With PCA

SVN doesn't like it when columns are highly correlated. 

In [ ]:
pca_cols = [
    'total_enrolled',
    'total_credited',
    'total_approved',
    'total_evaluations',
    'weighted_avg_grade'
]

other_numeric_cols = [col for col in numeric_cols if col not in pca_cols]

preprocessor3 =  ColumnTransformer(
    transformers=[
        ('pca_group', Pipeline([ 
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95))
        ]), pca_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
        
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('standardizer', StandardScaler())
        ]), other_numeric_cols)
    ],
    remainder='drop'
)

svm0_pipe = Pipeline([
    ('preprocessor', preprocessor3),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True))
])

svm0_pipe.fit(X_train, y_train)

# Show model metrics
eval_df = evaluate_model(svm0_pipe, model_name="SVM0")


In [ ]:
# Use t-SNE to visualize data in 2 dimensions
from sklearn.manifold import TSNE


X_train_preprocessed = preprocessor3.fit_transform(X_train)
X_test_preprocessed = preprocessor3.transform(X_test)

X_tsne = TSNE(n_components=2, random_state=13, perplexity=30).fit_transform(X_train_preprocessed)
tsne_df = pd.DataFrame(X_tsne, columns=['Dim1','Dim2'])
tsne_df['target'] = y_train.values

# now plot
sns.scatterplot(data=tsne_df, x='Dim1', y='Dim2', hue='target', palette='coolwarm', s=5)
plt.title("Figure 3:4: t-SNE of Preprocessed Training Data")
plt.show()


In [ ]:
# use cross validation to tune

kernels = ['linear', 'rbf']
Cs = [0.1, 1, 10]
gammas = ['scale', 0.1, 0.01]

svc_results = []

# test our tuning parameters
for test_kernel, test_C, test_gamma in product(kernels, Cs, gammas):

    pipe = ImbPipeline([
        ('preprocessor', preprocessor3),
        ('svm', SVC(kernel=test_kernel, C=test_C, gamma=test_gamma))
    ])

    # get accuracy and f1 score using 5-fold CV
    scores = cross_validate(
        pipe,
        X_train, y_train,
        cv=5,
        scoring={
            'accuracy': 'accuracy', 
            'recall_yes': make_scorer(recall_score, pos_label=1), 
            'f1_yes': make_scorer(f1_score, pos_label=1)
        },
        n_jobs=-1
    )

    svc_results.append({
        'kernel': test_kernel,
        'C': test_C,
        'gamma': test_gamma,
        'accuracy': scores['test_accuracy'].mean(),
        'recall_yes': scores['test_recall_yes'].mean(),
        'f1_mean': scores['test_f1_yes'].mean()
    })


In [ ]:
# test our tuning parameters with SMOTE
for test_kernel, test_C, test_gamma in product(kernels, Cs, gammas):

    pipe = ImbPipeline([
        ('preprocessor', preprocessor3),
        ('smote', SMOTE(random_state=13)),
        ('svm', SVC(kernel=test_kernel, C=test_C, gamma=test_gamma))
    ])

    # get accuracy and f1 score using 5-fold CV
    scores = cross_validate(
        pipe,
        X_train, y_train,
        cv=5,
        scoring={
            'accuracy': 'accuracy', 
            'recall_yes': make_scorer(recall_score, pos_label=1), 
            'f1_yes': make_scorer(f1_score, pos_label=1)
        },
        n_jobs=-1
    )

    svc_results.append({
        'kernel': test_kernel + " w.SMOTE",
        'C': test_C,
        'gamma': test_gamma,
        'accuracy': scores['test_accuracy'].mean(),
        'recall_yes': scores['test_recall_yes'].mean(),
        'f1_mean': scores['test_f1_yes'].mean()
    })


print(pd.DataFrame(svc_results))

In [ ]:
# Linear SVM
linear_pipe = ImbPipeline([
    ('preprocessing', preprocessor3),
    ('classifier', SVC(kernel='linear', C=1, probability=True))
])
linear_pipe.fit(X_train, y_train)

eval_df = evaluate_model(linear_pipe, "SVMlinear")


In [ ]:
# Mix-in Smote
linear_pipe_w_smote = ImbPipeline([
    ('preprocessing', preprocessor3),
    ('smote', SMOTE(random_state=13)),
    ('classifier', SVC(kernel='linear', C=1, probability=True))
])
linear_pipe_w_smote.fit(X_train, y_train)

eval_df = evaluate_model(linear_pipe_w_smote, "SVMlin w.SMOTE")


In [ ]:

# Best RBF found
rbf_pipe = ImbPipeline([
    ('preprocessing', preprocessor3),
    ('classifier', SVC(kernel='rbf', C=10, gamma=0.01, probability=True))
])
rbf_pipe.fit(X_train, y_train)
eval_df = evaluate_model(rbf_pipe, "SVM_RBF")

In [ ]:
# Best RBF with SMOTE
rbf_pipe = ImbPipeline([
    ('preprocessing', preprocessor3),
    ('smote', SMOTE(random_state=13)),
    ('classifier', SVC(kernel='rbf', C=10, gamma=0.01, probability=True))
])
rbf_pipe.fit(X_train, y_train)
eval_df = evaluate_model(rbf_pipe, "SVM_RBF w.SMOTE")

### 3.4 Neural Networds

In [ ]:
# NN Using PyTorch
preprocessor_nn = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('standardizer', StandardScaler())
        ]), numeric_cols)
    ], remainder='drop'
)


In [ ]:
# preprocess training and testing set
X_train_prep = preprocessor_nn.fit_transform(X_train)
X_test_prep  = preprocessor_nn.transform(X_test)


In [ ]:

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)
X_test_tensor = torch.tensor(X_test_prep, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1,1)



In [ ]:
# Define Simple NN with flexible number of hidden layers
class LogisticRegression(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(LogisticRegression, self).__init__()
        
        in_dim = input_size
        layers = []
        for h in hidden_units:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h

        layers.append(nn.Linear(in_dim, 1))

        self.network = nn.Sequential(*layers)


    def forward(self, x):
        return self.network(x)


In [ ]:
#  Helper to train using epochs in batches 

fold_history = []

y_train_flat = y_train_tensor.numpy().flatten()


def evalWithCV(hidden_size, learning_rate=0.01, num_epochs=50, silent=True):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor, y_train_flat)):

        X_train_kfold = X_train_tensor[train_idx]
        y_train_kfold = y_train_tensor[train_idx]

        X_val = X_train_tensor[val_idx]
        y_val = y_train_tensor[val_idx]

        train_loader = DataLoader(TensorDataset(X_train_kfold, y_train_kfold),
                                batch_size=32, shuffle=True)
        
        val_loader   = DataLoader(TensorDataset(X_val, y_val),
                                batch_size=32, shuffle=False)
        input_size = X_train_tensor.shape[1]
        model = LogisticRegression(input_size, hidden_size)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)


        loss_hist = []
        acc_hist = []

        # train using epochs across batches 
        for epoch in range(num_epochs):
            model.train()
            train_loss = 0
            for x_batch, y_batch in train_loader:
                pred = model(x_batch)
                loss = criterion(pred, y_batch)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad() 
                train_loss += loss.item() * x_batch.size(0)
            train_loss /= len(train_loader.dataset)
            loss_hist.append(train_loss)

            # validate model
            model.eval()
            val_loss = 0
            val_logits_list = []
            val_true_list = []

            with torch.no_grad():
                for x_val_batch, y_val_batch in val_loader:
                    logits = model(x_val_batch)
                    loss = criterion(logits, y_val_batch)
                    val_loss += loss.item() * x_val_batch.size(0)
                    val_logits_list.append(logits)
                    val_true_list.append(y_val_batch)

            val_loss /= len(val_loader.dataset)
            val_logits = torch.cat(val_logits_list, dim=0)
            val_probs  = torch.sigmoid(val_logits).numpy().flatten()
            val_true   = torch.cat(val_true_list, dim=0).numpy().flatten()
            val_pred = (val_probs >= 0.5).astype(int)
            val_acc = accuracy_score(val_true, val_pred)
            
            acc = val_acc
            prec, rec, f1, _ = precision_recall_fscore_support(val_true, val_pred, average='binary', zero_division=0)
            auc = roc_auc_score(val_true, val_probs)
            auprc = average_precision_score(val_true, val_probs)

            fold_metrics.append((val_loss, acc, prec, rec, f1, auc, auprc))
            acc_hist.append(f1)

            if (silent == False):
                print(f"E{epoch+1} / F:{fold+1} — Loss: {val_loss:.4g}, ACC: {acc:.4f}, PREC: {prec:.4f}, REC: {rec:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

        fold_history.append({
            "train_loss": loss_hist,
            "val_acc": acc_hist
        }) 

    return fold_metrics

In [ ]:
# Helper for printing metrics
def printMetrics(mean_scores, std_scores):
    print(f"Loss: {mean_scores[0]:.4f} ± {std_scores[0]:.4f}")
    print(f"Accuracy: {mean_scores[1]:.4f} ± {std_scores[1]:.4f}")
    print(f"Precision: {mean_scores[2]:.4f} ± {std_scores[2]:.4f}")
    print(f"Recall: {mean_scores[3]:.4f} ± {std_scores[3]:.4f}")
    print(f"F1: {mean_scores[4]:.4f} ± {std_scores[4]:.4f}")
    print(f"AUC: {mean_scores[5]:.4f} ± {std_scores[5]:.4f}")
    print(f"AUPRC: {mean_scores[6]:.4f} ± {std_scores[6]:.4f}")


In [ ]:
# tain and evaluate NN model with 1 hidden layer and learning rate of 0.01
fold_metrics_nn0_raw = evalWithCV([32])


# print K-fold averages
fold_metrics_nn0 = np.array(fold_metrics_nn0_raw)

# print K-fold averages
mean_scores_0 = fold_metrics_nn0.mean(axis=0)
std_scores_0  = fold_metrics_nn0.std(axis=0)
printMetrics(mean_scores_0, std_scores_0)


In [ ]:

fold_metrics_nn1_raw = evalWithCV([32], 1e-3, 50)

# print K-fold averages
fold_metrics_nn1 = np.array(fold_metrics_nn1_raw)

# print K-fold averages
mean_scores_1 = fold_metrics_nn1.mean(axis=0)
std_scores_1  = fold_metrics_nn1.std(axis=0)
printMetrics(mean_scores_1, std_scores_1)

In [ ]:
# tain and evaluate NN model with 2 hidden layer
fold_metrics_nn2_raw = evalWithCV([32], 1e-4)

# print K-fold averages
fold_metrics_nn2 = np.array(fold_metrics_nn2_raw)

# print K-fold averages
mean_scores_2 = fold_metrics_nn2.mean(axis=0)
std_scores_2  = fold_metrics_nn2.std(axis=0)

printMetrics(mean_scores_2, std_scores_2)


In [ ]:
# tain and evaluate NN model with 2 hidden layer
fold_metrics_nn3_raw = evalWithCV([64, 32])

# print K-fold averages
fold_metrics_nn3 = np.array(fold_metrics_nn3_raw)

# print K-fold averages
mean_scores_3 = fold_metrics_nn3.mean(axis=0)
std_scores_3  = fold_metrics_nn3.std(axis=0)

printMetrics(mean_scores_3, std_scores_3)

In [ ]:
# tain and evaluate NN model with 2 hidden layer
fold_metrics_nn4_raw = evalWithCV([64, 32], 1e-3)

# print K-fold averages
fold_metrics_nn4 = np.array(fold_metrics_nn4_raw)

# print K-fold averages
mean_scores_4 = fold_metrics_nn4.mean(axis=0)
std_scores_4  = fold_metrics_nn4.std(axis=0)

printMetrics(mean_scores_4, std_scores_4)

In [ ]:
# tain and evaluate NN model with 2 hidden layer
fold_metrics_nn5_raw = evalWithCV([64, 32], 1e-4)

# print K-fold averages
fold_metrics_nn5 = np.array(fold_metrics_nn5_raw)

# print K-fold averages
mean_scores_5 = fold_metrics_nn5.mean(axis=0)
std_scores_5  = fold_metrics_nn5.std(axis=0)

printMetrics(mean_scores_5, std_scores_5)

In [ ]:
# Plot evaluated models

fig = plt.figure(figsize=(12,5))
ax = fig.add_subplot(1, 2, 1)
ax.plot(fold_history[0]['train_loss'], color="red", label="Model 0 Loss")
ax.plot(fold_history[1]['train_loss'], color="green", label="Model 1 Loss")
ax.plot(fold_history[2]['train_loss'], color="black", label="Model 2 Loss")
ax.set_title('Training Loss (1x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()

ax = fig.add_subplot(1, 2, 2)
ax.plot(fold_history[0]['val_acc'], lw=2, color="red", label="Model 0 F1")
ax.plot(fold_history[1]['val_acc'], lw=2, color="green", label="Model 1 F1")
ax.plot(fold_history[2]['val_acc'], lw=2, color="black", label="Model 2 F1")
ax.set_title('Training F1 (1x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()

plt.show()


fig = plt.figure(figsize=(12,5))

ax = fig.add_subplot(1, 2, 1)
ax.plot(fold_history[3]['train_loss'], color="orange", label="Model 3 Loss")
ax.plot(fold_history[4]['train_loss'], color="brown", label="Model 4 Loss")
ax.plot(fold_history[5]['train_loss'], color="blue", label="Model 5 Loss")
ax.set_title('Training Loss (2x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()

ax = fig.add_subplot(1, 2, 2)
ax.plot(fold_history[3]['val_acc'], lw=2, color="orange", label="Model 3 F1")
ax.plot(fold_history[4]['val_acc'], lw=2, color="brown", label="Model 4 F1")
ax.plot(fold_history[5]['val_acc'], lw=2, color="blue", label="Model 5 F1")
ax.set_title('Training F1 (2x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()

plt.show()

fig = plt.figure(figsize=(12,5))

ax = fig.add_subplot(1, 2, 1)
ax.plot(fold_history[1]['train_loss'], color="green", label="Model 1 Loss")
ax.plot(fold_history[5]['train_loss'], color="blue", label="Model 5 Loss")
ax.set_title('Training Loss (1x HL vs 2x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()

ax = fig.add_subplot(1, 2, 2)
ax.plot(fold_history[1]['val_acc'], lw=2, color="green", label="Model 1 F1")
ax.plot(fold_history[5]['val_acc'], lw=2, color="blue", label="Model 5 F1")
ax.set_title('Training F1 (1x HL vs 2x HL)', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.legend()


In [ ]:
# Train NN model with full tensor dataset
seed = 13
np.random.seed(seed)
torch.manual_seed(seed)

nn_model1 = LogisticRegression(X_train_tensor.shape[1], [32])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model1.parameters(), lr=1e-3, weight_decay=1e-4)


train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                                batch_size=32, shuffle=True)

# train using epochs across batches 
for epoch in range(50):

    for x_batch, y_batch in train_loader:
        pred = nn_model1(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() 

In [ ]:
nn_model1.eval()
with torch.no_grad():
    logits = nn_model1(X_test_tensor)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
    preds = (probs >= 0.5).astype(int)

y_true = y_test_tensor.cpu().numpy().flatten()

print(f'Confusion Matrix for NN1\n', confusion_matrix(y_test, y_pred))
print(f"\nClassification Report for NN1:\n", classification_report(y_test, y_pred))

# eval_df = eval_df.drop(df[df["Model"] == "NN1"].index)
eval_df = append_eval("NN1", y_true, preds, probs)



In [ ]:
print(X_train_tensor.shape)

In [ ]:
# Train NN model with full tensor dataset
seed = 13
np.random.seed(seed)
torch.manual_seed(seed)

nn_model2 = LogisticRegression(X_train_tensor.shape[1], [64, 32])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model2.parameters(), lr=1e-4, weight_decay=1e-4)

# train using epochs across batches 
for epoch in range(50):
    for x_batch, y_batch in train_loader:
        pred = nn_model2(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() 

In [ ]:
nn_model5.eval()
with torch.no_grad():
    logits = nn_model5(X_test_tensor)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
    preds = (probs >= 0.5).astype(int)

y_true = y_test_tensor.cpu().numpy().flatten()

print(f'Confusion Matrix for NN1\n', confusion_matrix(y_test, y_pred))
print(f"\nClassification Report for NN1:\n", classification_report(y_test, y_pred))

eval_df = append_eval("NN5", y_true, preds, probs)



In [ ]:

print(eval_df)


In [ ]:
# Use SHAP to summarize feature importance in NN model

import shap

num_features = numeric_cols
ohe = preprocessor2.named_transformers_['cat'].named_steps['encoder']
cat_features = ohe.get_feature_names_out(categorical_cols)
feature_names = np.concatenate([num_features, cat_features])

background = X_train_tensor[:100]
X_eval = X_test_tensor[:50] 

explainer = shap.DeepExplainer(nn_model1, background)
shap_values = explainer.shap_values(X_eval)

if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_vals = shap_values.squeeze(-1)
else:
    shap_vals = shap_values

print("SHAP values shape:", shap_vals.shape)  #


shap.summary_plot(
    shap_vals,
    X_eval.numpy(),
    feature_names=feature_names,
    plot_type="dot"
)


In [ ]:
print(X_train_tensor.shape)